<a href="https://colab.research.google.com/github/moshuhuang/RouteGuard-last-mile-address-risk/blob/main/02_feature_engineering_and_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Git Projects/address_risk_dataset_30k.csv')
print(df.shape)
print(df['is_address_risk'].value_counts())
print(df.head(3))

Mounted at /content/drive
(30000, 16)
is_address_risk
0    24000
1     6000
Name: count, dtype: int64
               hash  number                 street unit           city  \
0  8c1540842546b742    8239       CLAW GLADES LOOP  NaN  WESLEY CHAPEL   
1  a253306587832a86   33333           ARTHUR DRIVE  NaN  WESLEY CHAPEL   
2  9a6acf4b00f7cc09    9136  BELLEVIEW FOREST PASS  NaN    ZEPHYRHILLS   

   district  region  postcode  id  accuracy  longitude   latitude  \
0       NaN     NaN     33545 NaN       NaN -82.296992  28.277444   
1       NaN     NaN     33543 NaN       NaN -82.264688  28.212970   
2       NaN     NaN     33541 NaN       NaN -82.241848  28.290200   

                                    original_address  \
0     8239 CLAW GLADES LOOP, WESLEY CHAPEL, FL 33545   
1        33333 ARTHUR DRIVE, WESLEY CHAPEL, FL 33543   
2  9136 BELLEVIEW FOREST PASS, ZEPHYRHILLS, FL 33541   

                                       input_address error_type  \
0     8239 CLAW GLADES LOOP, WES

In [2]:
import re

def engineer_features(df):
    addr = df['input_address'].fillna('')

    # 长度类
    df['address_length'] = addr.str.len()
    df['word_count'] = addr.str.split().str.len()
    df['digit_count'] = addr.str.count(r'\d')
    df['comma_count'] = addr.str.count(',')
    df['special_char_count'] = addr.str.count(r'[^a-zA-Z0-9\s,]')

    # 结构信号
    df['has_house_number'] = addr.str.match(r'^\d+').astype(int)
    df['has_unit'] = addr.str.contains(
        r'\b(APT|UNIT|STE|SUITE|#|LOT|BLDG)\b', case=False).astype(int)
    df['has_zip'] = addr.str.contains(r'\b\d{5}\b').astype(int)
    df['has_city'] = addr.str.contains(r',\s*[A-Z]').astype(int)
    df['is_street_only'] = (
        (df['has_house_number'] == 0) &
        (df['has_zip'] == 0)
    ).astype(int)
    df['unit_keyword_no_number'] = (
        addr.str.contains(r'\b(APT|UNIT|STE|SUITE)\b', case=False) &
        ~addr.str.contains(r'\b(APT|UNIT|STE|SUITE)\s*\d+', case=False)
    ).astype(int)
    df['has_ambiguous_abbrev'] = addr.str.contains(
        r'\bWC\b|\bNC\b|\bEC\b|\bSC\b', case=False).astype(int)

    return df

df = engineer_features(df)

feature_cols = [
    'address_length', 'word_count', 'digit_count', 'comma_count',
    'special_char_count', 'has_house_number', 'has_unit', 'has_zip',
    'has_city', 'is_street_only', 'unit_keyword_no_number', 'has_ambiguous_abbrev'
]

print(df[feature_cols].describe())
print("\n缺失值:", df[feature_cols].isnull().sum().sum())

/tmp/ipykernel_8393/4104862050.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['has_unit'] = addr.str.contains(
/tmp/ipykernel_8393/4104862050.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  addr.str.contains(r'\b(APT|UNIT|STE|SUITE)\b', case=False) &
/tmp/ipykernel_8393/4104862050.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~addr.str.contains(r'\b(APT|UNIT|STE|SUITE)\s*\d+', case=False)


       address_length    word_count   digit_count   comma_count  \
count    30000.000000  30000.000000  30000.000000  30000.000000   
mean        43.045733      7.391500      9.435700      1.950367   
std          6.708557      1.370045      2.039282      0.272468   
min          9.000000      2.000000      0.000000      0.000000   
25%         40.000000      6.000000      9.000000      2.000000   
50%         43.000000      7.000000     10.000000      2.000000   
75%         47.000000      8.000000     10.000000      2.000000   
max         65.000000     12.000000     16.000000      2.000000   

       special_char_count  has_house_number      has_unit       has_zip  \
count        30000.000000      30000.000000  30000.000000  30000.000000   
mean             0.007633          0.959000      0.126633      0.967767   
std              0.088178          0.198293      0.332567      0.176622   
min              0.000000          0.000000      0.000000      0.000000   
25%              0.00

In [3]:
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df['is_address_risk']

# First split off test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Split remaining into train (70%) and val (15%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train risk rate: {y_train.mean():.3f}")
print(f"Val risk rate:   {y_val.mean():.3f}")
print(f"Test risk rate:  {y_test.mean():.3f}")

Train: 21012 | Val: 4488 | Test: 4500
Train risk rate: 0.200
Val risk rate:   0.200
Test risk rate:  0.200


## Rule Based Model

In [4]:
def rule_based_predict(df):
    risk = (
        (df['has_zip'] == 0) |
        (df['has_house_number'] == 0) |
        (df['is_street_only'] == 1) |
        (df['unit_keyword_no_number'] == 1)
    ).astype(int)
    return risk

from sklearn.metrics import classification_report, roc_auc_score

y_pred_rule = rule_based_predict(X_val)
print(classification_report(y_val, y_pred_rule))
print("AUC:", roc_auc_score(y_val, y_pred_rule))

              precision    recall  f1-score   support

           0       0.85      0.99      0.92      3590
           1       0.93      0.30      0.46       898

    accuracy                           0.86      4488
   macro avg       0.89      0.65      0.69      4488
weighted avg       0.87      0.86      0.82      4488

AUC: 0.6483835946175655


## Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Scale features - LR is sensitive to feature scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train
lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Evaluate
y_pred_lr = lr.predict(X_val_scaled)
y_prob_lr = lr.predict_proba(X_val_scaled)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_val, y_pred_lr))
print("AUC:", roc_auc_score(y_val, y_prob_lr))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.87      0.94      0.90      3590
           1       0.64      0.44      0.52       898

    accuracy                           0.84      4488
   macro avg       0.76      0.69      0.71      4488
weighted avg       0.82      0.84      0.83      4488

AUC: 0.7381766041528373


## XGBOOST

In [6]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight to handle class imbalance
# = number of negative samples / number of positive samples
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Train
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train, y_train)

# Evaluate
y_pred_xgb = xgb.predict(X_val)
y_prob_xgb = xgb.predict_proba(X_val)[:, 1]

print("\n=== XGBoost (Structured Features) ===")
print(classification_report(y_val, y_pred_xgb))
print("AUC:", roc_auc_score(y_val, y_prob_xgb))

scale_pos_weight: 4.00

=== XGBoost (Structured Features) ===
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      3590
           1       0.80      0.41      0.54       898

    accuracy                           0.86      4488
   macro avg       0.83      0.69      0.73      4488
weighted avg       0.85      0.86      0.84      4488

AUC: 0.7300778269258209


## Text Embedding

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for input_address
print("Generating embeddings...")
train_idx = X_train.index
val_idx = X_val.index
test_idx = X_test.index

train_embeddings = embedder.encode(df.loc[train_idx, 'input_address'].tolist(),
                                    batch_size=256, show_progress_bar=True)
val_embeddings = embedder.encode(df.loc[val_idx, 'input_address'].tolist(),
                                  batch_size=256, show_progress_bar=True)
test_embeddings = embedder.encode(df.loc[test_idx, 'input_address'].tolist(),
                                   batch_size=256, show_progress_bar=True)

print(f"Embedding shape: {train_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Generating embeddings...


Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Embedding shape: (21012, 384)


## XGBoost (Structured + Embedding Fusion)

In [8]:
# Combine structured features + embeddings
X_train_fused = np.hstack([X_train.values, train_embeddings])
X_val_fused = np.hstack([X_val.values, val_embeddings])
X_test_fused = np.hstack([X_test.values, test_embeddings])

print(f"Fused feature shape: {X_train_fused.shape}")

# Train
xgb_fused = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_fused.fit(X_train_fused, y_train)

# Evaluate
y_pred_fused = xgb_fused.predict(X_val_fused)
y_prob_fused = xgb_fused.predict_proba(X_val_fused)[:, 1]

print("\n=== XGBoost (Structured + Embedding) ===")
print(classification_report(y_val, y_pred_fused))
print("AUC:", roc_auc_score(y_val, y_prob_fused))

Fused feature shape: (21012, 396)

=== XGBoost (Structured + Embedding) ===
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      3590
           1       0.79      0.48      0.59       898

    accuracy                           0.87      4488
   macro avg       0.83      0.72      0.76      4488
weighted avg       0.86      0.87      0.86      4488

AUC: 0.7489394569175698


## Threshold Optimization

In [9]:
from sklearn.metrics import precision_recall_curve
import numpy as np

# Find optimal threshold for F1 on fused model
precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob_fused)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Default threshold (0.5) → Recall: 0.48, F1: 0.59")
print(f"Optimal threshold: {best_threshold:.3f}")

y_pred_optimal = (y_prob_fused >= best_threshold).astype(int)
print("\n=== XGBoost Fused (Optimal Threshold) ===")
print(classification_report(y_val, y_pred_optimal))

Default threshold (0.5) → Recall: 0.48, F1: 0.59
Optimal threshold: 0.524

=== XGBoost Fused (Optimal Threshold) ===
              precision    recall  f1-score   support

           0       0.88      0.98      0.93      3590
           1       0.84      0.47      0.60       898

    accuracy                           0.88      4488
   macro avg       0.86      0.72      0.76      4488
weighted avg       0.87      0.88      0.86      4488



## SMOTE + XGBoost

In [10]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_fused_sm, y_train_sm = sm.fit_resample(X_train_fused, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}")

xgb_smote = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_smote.fit(X_train_fused_sm, y_train_sm)

y_prob_smote = xgb_smote.predict_proba(X_val_fused)[:, 1]
y_pred_smote = (y_prob_smote >= 0.4).astype(int)  # lower threshold

print("\n=== XGBoost Fused + SMOTE ===")
print(classification_report(y_val, y_pred_smote))
print("AUC:", roc_auc_score(y_val, y_prob_smote))

Before SMOTE: {0: 16810, 1: 4202}
After SMOTE: {0: 16810, 1: 16810}

=== XGBoost Fused + SMOTE ===
              precision    recall  f1-score   support

           0       0.88      0.83      0.86      3590
           1       0.45      0.56      0.50       898

    accuracy                           0.78      4488
   macro avg       0.67      0.69      0.68      4488
weighted avg       0.80      0.78      0.78      4488

AUC: 0.7322970885471274


## Final Evaluation on Test Set

In [11]:
# Final model: XGBoost Fused (best val F1 and AUC)
y_pred_test = xgb_fused.predict(X_test_fused)
y_prob_test = xgb_fused.predict_proba(X_test_fused)[:, 1]

print("=" * 50)
print("FINAL MODEL EVALUATION ON HELD-OUT TEST SET")
print("Model: XGBoost (Structured Features + Text Embedding)")
print("=" * 50)
print(classification_report(y_test, y_pred_test))
print("AUC:", roc_auc_score(y_test, y_prob_test))

# Summary table
print("\n=== Model Comparison Summary (Validation Set) ===")
results = {
    'Model': ['Rule-based', 'Logistic Regression',
              'XGBoost (Structured)', 'XGBoost (Fused)', 'XGBoost + SMOTE'],
    'Recall': [0.30, 0.44, 0.41, 0.48, 0.56],
    'F1':     [0.46, 0.52, 0.54, 0.59, 0.50],
    'AUC':    [0.65, 0.74, 0.73, 0.75, 0.73]
}
print(pd.DataFrame(results).to_string(index=False))
print("\nFinal model selected: XGBoost (Fused) — best F1 (0.59) and AUC (0.75)")

FINAL MODEL EVALUATION ON HELD-OUT TEST SET
Model: XGBoost (Structured Features + Text Embedding)
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      3600
           1       0.81      0.52      0.64       900

    accuracy                           0.88      4500
   macro avg       0.85      0.75      0.78      4500
weighted avg       0.87      0.88      0.87      4500

AUC: 0.7735077160493827

=== Model Comparison Summary (Validation Set) ===
               Model  Recall   F1  AUC
          Rule-based    0.30 0.46 0.65
 Logistic Regression    0.44 0.52 0.74
XGBoost (Structured)    0.41 0.54 0.73
     XGBoost (Fused)    0.48 0.59 0.75
     XGBoost + SMOTE    0.56 0.50 0.73

Final model selected: XGBoost (Fused) — best F1 (0.59) and AUC (0.75)


# New Section